# Superstore Sales Analysis

**Author:** Eman Mahmoud Ahmed Ali Omar
**Mentor:** Ali Mohamed — Gulf Plastics Industries (GPI)
**Date:** 2026-09-16
**Dataset:** [Superstore Sales Dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final) (9,994 rows, 2014-01-03 to 2017-12-30)

## Project Objective

This project analyzes four years of Superstore sales to identify which products, regions, and customer segments drive the most revenue and profit, and to flag areas where sales and profit diverge — specifically, where high sales come with low or negative profit.

## Analysis Questions

1. **Which product categories and regions generate the most revenue and profit?** *(bar chart)*
2. **How have sales and profit changed over time (monthly/yearly)?** *(line chart)*
3. **How does discount level relate to profit?** *(scatter plot + correlation)*

## Tools

Python, pandas, matplotlib, seaborn, sqlite3

> Checklist: All imports at the top (item 4), no hardcoded absolute paths (item 5), runs top-to-bottom on fresh clone (item 3).

## Section 1: Setup

Importing required libraries and setting style for all charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set_style("whitegrid")
%matplotlib inline

# Consistent style for all charts
plt.rcParams["figure.dpi"] = 150

## Section 2: Load Cleaned Data

Loading the cleaned dataset. If `data/superstore_clean.csv` is not present, falls back to loading raw data and applying minimal cleaning inline.

In [ ]:
# Load cleaned data (preferred)
try:
    df = pd.read_csv("data/superstore_clean.csv", parse_dates=["Order Date", "Ship Date"])
except FileNotFoundError:
    # Fallback: load raw and apply minimal cleaning inline
    df = pd.read_csv("data/superstore.csv", encoding="latin-1")
    df["Order Date"] = pd.to_datetime(df["Order Date"])
    df["Ship Date"] = pd.to_datetime(df["Ship Date"])
    for col in ["Category", "Sub-Category", "Region", "Segment"]:
        df[col] = df[col].astype(str).str.strip()

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Section 3: Quick Overview of the Data

Confirming data types, summary statistics, and that no missing values remain.

In [ ]:
print(df.dtypes)
print()
print(df.describe())
print()
print(df.isnull().sum())
print()
print(f"Date range: {df['Order Date'].min()} to {df['Order Date'].max()}")

## Section 4: Question 1 — Which product categories and regions generate the most revenue and profit?

> **Why it matters:** Identifying top-performing categories and regions tells the business where to focus inventory, marketing, and sales effort, and reveals whether high sales come with low profit. Technology leads in both sales and profit, while Furniture generates high revenue but near-zero profit — a critical business insight.

### 4a. Compute the answer

Total sales and profit by Category and Region, sorted descending, with descriptive statistics.

In [ ]:
# Q1: Compute total sales and profit by Category and Region
cat_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
cat_profit = df.groupby("Category")["Profit"].sum().sort_values(ascending=False)
region_sales = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
region_profit = df.groupby("Region")["Profit"].sum().sort_values(ascending=False)

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print()
print("=== SALES BY CATEGORY ===")
for cat, val in cat_sales.items():
    print(f"  {cat}: ${val:,.2f} ({val/total_sales*100:.1f}% of total)")
print()
print("=== PROFIT BY CATEGORY ===")
for cat, val in cat_profit.items():
    print(f"  {cat}: ${val:,.2f} ({val/total_profit*100:.1f}% of total)")
print()
print("=== SALES BY REGION ===")
for reg, val in region_sales.items():
    print(f"  {reg}: ${val:,.2f} ({val/total_sales*100:.1f}% of total)")
print()
print("=== PROFIT BY REGION ===")
for reg, val in region_profit.items():
    print(f"  {reg}: ${val:,.2f} ({val/total_profit*100:.1f}% of total)")

# Descriptive stats for context
print("\n=== Descriptive Stats — Sales ===")
print(df["Sales"].describe())
print("\n=== Descriptive Stats — Profit ===")
print(df["Profit"].describe())

# Profit margin by category
print("\n=== Profit Margin by Category ===")
for cat in cat_sales.index:
    cat_df = df[df["Category"]==cat]
    margin = cat_df["Profit"].sum() / cat_df["Sales"].sum() * 100
    print(f"  {cat}: {margin:.1f}%")

### 4b. Visualize

Bar charts of total sales and profit by category. Key insight: Technology dominates profit while Furniture has high sales but almost no profit.

In [ ]:
# Q1: Bar chart — Sales and Profit by Category
colors = ["#2196F3", "#4CAF50", "#FF9800"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_sales.plot(kind="bar", ax=axes[0], color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Total Sales by Product Category", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Total Sales ($)")
for i, (cat, val) in enumerate(cat_sales.items()):
    axes[0].text(i, val + 15000, f"${val/1000:.0f}K\n({val/total_sales*100:.0f}%)", ha="center", fontsize=9)

cat_profit.plot(kind="bar", ax=axes[1], color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_title("Total Profit by Product Category", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Total Profit ($)")
for i, (cat, val) in enumerate(cat_profit.items()):
    axes[1].text(i, val + 3000, f"${val/1000:.0f}K\n({val/total_profit*100:.0f}%)", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("images/question1_revenue_by_category.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: images/question1_revenue_by_category.png")

### 4c. Interpret

> **Finding:** Technology generated **$836,154** in sales (36.4% of total revenue) and **$145,455** in profit (50.8% of total profit), with a 17.4% profit margin. Furniture had the second-highest sales at $742,000 (32.3% of total) but only generated **$18,451** in profit — a margin of just **2.5%**. Office Supplies earned $719,047 (31.3% of sales) with $122,491 profit (42.8%, 17.0% margin).
>
> **Business meaning:** Furniture is a revenue trap — it brings in nearly as much revenue as Technology but contributes almost nothing to profit. The business should investigate Furniture pricing, costs, or discount strategy. Technology is the clear profit engine.
>
> **Regional pattern:** West leads both sales ($725,458) and profit ($108,418), while Central lags in both ($501,240 sales, $39,706 profit). South has the lowest sales but a surprising 16.3% profit share.

## Section 5: Question 2 — How have sales and profit changed over time?

> **Why it matters:** Trend over 4 years shows whether the business is growing, shrinking, or seasonal — affecting forecasting and planning decisions.

### 5a. Compute the answer

Monthly and yearly aggregation of Sales and Profit, plus growth calculations.

In [ ]:
# Q2: Monthly and yearly aggregation
df["Year"] = df["Order Date"].dt.year
df["Quarter"] = df["Order Date"].dt.quarter
monthly = df.groupby(df["Order Date"].dt.to_period("M"))["Sales"].sum()
yearly = df.groupby("Year")["Sales", "Profit"].sum()
quarterly = df.groupby(["Year", "Quarter"])["Sales"].sum().unstack()

print("=== YEARLY SALES & PROFIT ===")
for yr, row in yearly.iterrows():
    print(f"  {yr}: Sales=${row['Sales']:,.2f}, Profit=${row['Profit']:,.2f}")

sales_2014 = yearly.loc[2014, "Sales"]
sales_2017 = yearly.loc[2017, "Sales"]
print(f"\nSales growth 2014→2017: {(sales_2017-sales_2014)/sales_2014*100:.1f}%")

print("\n=== QUARTERLY SALES ===")
print(quarterly.round(0))

q4_mean = df[df["Quarter"]==4]["Sales"].mean()
other_mean = df[df["Quarter"]!=4]["Sales"].mean()
print(f"\nQ4 avg/order: ${q4_mean:,.2f}")
print(f"Other quarters avg/order: ${other_mean:,.2f}")
print(f"Q4 premium: {((q4_mean/other_mean)-1)*100:.1f}%")

### 5b. Visualize

Line chart of monthly sales and profit showing the clear upward trend and Q4 seasonality.

In [ ]:
# Q2: Line chart — Monthly Sales and Profit Trend
fig, ax = plt.subplots(figsize=(12, 5))
monthly = df.groupby(df["Order Date"].dt.to_period("M"))["Sales", "Profit"].sum()
monthly["Sales"].plot(ax=ax, label="Sales", linewidth=2, color="#2196F3")
monthly["Profit"].plot(ax=ax, label="Profit", linewidth=2, color="#4CAF50")
ax.set_title("Monthly Sales and Profit Trend (2014-2017)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Amount ($)")
ax.legend(loc="upper left", fontsize=11)
plt.tight_layout()
plt.savefig("images/question2_sales_over_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: images/question2_sales_over_time.png")

### 5c. Interpret

> **Finding:** Sales grew **51.4%** from $484,247 in 2014 to $733,215 in 2017, with profit growing 88.7% from $49,544 to $93,439. Every year shows a consistent **Q4 peak** — Q4 2017 sales reached $280,054, nearly double Q1 2014 ($74,448). The Q4 seasonal premium is **5.3% higher** per order than other quarters.
>
> **Business meaning:** The business is clearly growing and has strong holiday seasonality. Q4 should be the focus for inventory and staffing. The profit growth rate (88.7%) exceeding sales growth (51.4%) suggests improving operational efficiency.
>
> **Caution:** This 4-year trend may not continue. The dataset does not include 2018+ data.

## Section 6: Question 3 — How does discount level relate to profit?

> **Why it matters:** Discounts drive volume but can erode margins. Understanding this relationship helps the business set discount policy without unintentionally destroying profit.

### 6a. Compute the answer

Correlation coefficient between Discount and Profit, plus grouped comparison by discount bracket.

In [ ]:
# Q3: Correlation and binned analysis
corr = df[["Discount", "Profit"]].corr().iloc[0, 1]
print(f"Correlation (Discount, Profit): {corr:.4f}")

# Bin discounts
bins = [-0.01, 0, 0.2, 0.4, 0.8]
labels = ["0%", "0-20%", "20-40%", "40%+"]
df["Disc_Bin"] = pd.cut(df["Discount"], bins=bins, labels=labels)
bin_stats = df.groupby("Disc_Bin")["Profit"].agg(["mean", "median", "count"])

print("\n=== PROFIT BY DISCOUNT BIN ===")
for idx, row in bin_stats.iterrows():
    print(f"  {idx} discount: avg=${row['mean']:,.2f}, median=${row['median']:,.2f}, n={int(row['count'])}")

disc_0 = df[df["Discount"]==0]["Profit"].mean()
disc_gt20 = df[df["Discount"]>0.2]["Profit"].mean()
print(f"\nAvg profit 0% discount: ${disc_0:,.2f}")
print(f"Avg profit >20% discount: ${disc_gt20:,.2f}")
print(f"Difference: ${disc_0-disc_gt20:,.2f}")

### 6b. Visualize

Scatter plot of discount vs profit showing the negative relationship, plus average profit by discount bracket.

In [ ]:
# Q3: Scatter plot and bar chart
corr = df[["Discount", "Profit"]].corr().iloc[0, 1]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["Discount"], df["Profit"], alpha=0.15, s=5, color="#2196F3", edgecolors="none")
axes[0].set_title(f"Discount vs Profit (r = {corr:.3f})", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Discount (%)")
axes[0].set_ylabel("Profit ($)")
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=1, alpha=0.5)
axes[0].axvline(x=0.2, color="red", linestyle="--", linewidth=1, alpha=0.5)
axes[0].text(0.21, df["Profit"].max()*0.9, "20% threshold", color="red", fontsize=9)

bins = [-0.01, 0, 0.2, 0.4, 0.8]
labels = ["0%", "0-20%", "20-40%", "40%+"]
df["Disc_Bin"] = pd.cut(df["Discount"], bins=bins, labels=labels)
bin_means = df.groupby("Disc_Bin")["Profit"].mean()
bin_colors = ["#4CAF50", "#FFC107", "#FF9800", "#F44336"]
bin_means.plot(kind="bar", ax=axes[1], color=bin_colors, edgecolor="black")
axes[1].set_title("Average Profit by Discount Level", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Discount Bracket")
axes[1].set_ylabel("Average Profit ($)")
axes[1].axhline(y=0, color="red", linestyle="--", linewidth=1)
for i, val in enumerate(bin_means.values):
    axes[1].text(i, val + (5 if val > 0 else -15), f"${val:.0f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("images/question3_discount_vs_profit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: images/question3_discount_vs_profit.png")

### 6c. Interpret

> **Finding:** There is a **negative correlation** between discount and profit (r = -0.220). Orders with **0% discount** averaged **$66.90** in profit, while orders with **>20% discount** averaged **-$97.18** in profit — a difference of **$164.08**. The 20-40% bracket averaged -$77.86 and the 40%+ bracket averaged -$106.71.
>
> **Business meaning:** Deep discounts are destroying profit. The 40%+ discount bracket (933 orders) consistently loses money. The business should cap discounts at 20% or implement conditional discounts tied to product margin.
>
> **Key insight:** The correlation is moderate (-0.22), not perfect, meaning discount is one factor among many. But the pattern is clear and actionable — high-discount orders systematically lose money.

## Section 7: Additional Analysis — Boxplot + Outliers

> **Purpose:** Identify outlier orders that disproportionately affect totals. Boxplot is the 4th chart type, supporting Q1-Q3 analysis. Scope remains at 3 questions — this section provides supporting insight only.

In [ ]:
# Boxplot + IQR outlier check — 4th chart type
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(x=df["Sales"], ax=axes[0], color="#2196F3", width=0.4)
axes[0].set_title("Distribution of Sales — Outliers Above Upper Fence", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Sales ($)")

Q1_s = df["Sales"].quantile(0.25)
Q3_s = df["Sales"].quantile(0.75)
IQR_s = Q3_s - Q1_s
upper = Q3_s + 1.5*IQR_s
outliers = df[df["Sales"] > upper]
axes[0].axvline(x=upper, color="red", linestyle="--", linewidth=1, label=f"Upper fence: ${upper:.0f}")
axes[0].legend()

sns.boxplot(x=df["Profit"], ax=axes[1], color="#FF9800", width=0.4)
axes[1].set_title("Distribution of Profit — Heavy Negative Tail", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Profit ($)")
axes[1].axvline(x=0, color="red", linestyle="--", linewidth=1)

plt.tight_layout()
plt.savefig("images/additional_sales_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Sales outliers: {len(outliers)} ({len(outliers)/len(df):.1%} of orders)")
print(f"Upper fence: ${upper:.2f}, total sales in outliers: ${outliers['Sales'].sum():,.0f}")
print(f"Outlier categories: {outliers['Category'].value_counts().head(3).to_dict()}")

# SQL check
conn = sqlite3.connect(":memory:")
df.to_sql("superstore", conn, index=False, if_exists="replace")
sql_result = pd.read_sql("SELECT Category, SUM(Sales) as total_sales FROM superstore GROUP BY Category ORDER BY total_sales DESC", conn)
print("\nSQL verification:")
print(sql_result)

## Section 8: Findings

> 5-8 findings grounded in specific numbers from the sections above. Keep in plain English.

**Findings:**

1. **Technology is the profit engine** — generated $836,154 in sales (36.4% of total) and $145,455 in profit (50.8% of total profit), with a 17.4% margin. Source: Q1.

2. **Furniture is a revenue trap** — $742,000 in sales (32.3% of total) but only $18,451 in profit (6.4% of total), a margin of just 2.5%. Source: Q1.

3. **Sales grew 51.4% from 2014 to 2017** — from $484,247 to $733,215, with consistent Q4 peaks every year. Q4 2017 hit $280,054. Source: Q2.

4. **High discounts destroy profit** — orders with >20% discount averaged -$97.18 profit vs $66.90 for no-discount orders (r = -0.220). Source: Q3.

5. **West leads in both sales and profit** — $725,458 sales and $108,418 profit, the strongest region across both metrics. Source: Q1.

6. **11.7% of orders are sales outliers** (above $499) totaling $1,477,483 — concentrated in Furniture (467), Technology (400), and Office Supplies (300). Source: Additional Analysis.

## Section 9: Limitations

> At least one honest paragraph describing what this analysis cannot tell you.

This analysis is limited by the dataset's 4-year span (2014-2017), which makes long-term trend conclusions unreliable — the 51.4% growth rate may not continue beyond the data window. Cost data (shipping, manufacturing, overhead) is not included, so we can only measure gross profit, not net margin or true profitability by segment. Geography is at the state level, not store or customer level, preventing granular location-based strategy. The discount-profit correlation (r=-0.22) is moderate, meaning discount is one factor among many affecting profit — customer segment, product category, and region all interact with discount effects.

## Section 10: Conclusion

> 2-3 paragraph summary tying questions and findings together. State the most important takeaway in plain English.

**Conclusion:** This analysis of 9,994 Superstore orders from 2014-2017 reveals three critical insights. First, Technology dominates profit (50.8% of total) while Furniture is a revenue trap with only 2.5% margin — the business should focus on optimizing Furniture pricing or reducing its costs. Second, the business is growing (51.4% sales increase) with strong Q4 seasonality, suggesting the need for inventory and staffing planning around holiday peaks. Third, deep discounts (>20%) systematically destroy profit, with those orders averaging -$97.18.

The most important takeaway: **high revenue does not equal high profit**. Furniture generates $742K in sales but only $18K in profit, while Technology generates $836K and $145K. The business must shift focus from revenue maximization to profit optimization — particularly by rethinking discount policy and investigating Furniture's cost structure.

With more time, I would investigate customer-segment profitability, product-level margin analysis, and whether the Furniture profit issue stems from specific sub-categories (Bookcases, Chairs, Tables) or across the board.

## Section 11: Quality Checklist Before Submission

- [x] Every section has a markdown heading
- [x] Every code cell has a one-line markdown explanation above it
- [x] Every chart has a title, axis labels, and 2-3 sentence Interpretation
- [x] At least one chart exported as PNG and saved in `images/`
- [x] Findings section has 6 numbered findings, each with a specific number
- [x] Limitations section has an honest paragraph
- [x] Notebook runs top-to-bottom without errors on a fresh clone (Kernel → Restart & Run All)

> Self-review against `Final_Project_Checklist.md` — all 36 items must be ticked.